# ⚽ Fase 1 — Video Annotato (MVP)

**Cosa fa questo notebook:** prendi una clip di calcio (broadcast/TV) e ti restituisce lo stesso video con:
- box su **giocatori**, **palla** e **arbitro**
- **colori** diversi per le due squadre
- un **numero (ID)** che segue ogni giocatore nel tempo

**Come si usa (sei un principiante, leggi qui):**
1. In alto a sinistra: menu `Runtime` → `Cambia tipo di runtime` → scegli **GPU (T4)** → Salva.
2. Esegui le celle **una alla volta** dall'alto verso il basso: clicca sulla cella e premi `Shift + Invio`.
3. Quando arrivi alla cella di **upload**, carica una clip corta (10–30 secondi vanno benissimo per iniziare).
4. Alla fine ti scarichi il video annotato.

**Nota Scout Lab:** questo notebook serve per debug visivo rapido. Per importare dati nel frontend,
usare `02_analisi_GPU_colab.ipynb`, che genera CSV posizioni, radar con `#Track ID`, zip importabile
e output compatibili con Identity Engine.

> 💡 Tieni le clip **corte** all'inizio: 30 secondi di video = elaborazione veloce. Una partita intera la affronteremo dopo.

## 1) Installazione librerie
Installiamo gli strumenti: `ultralytics` (modello che riconosce gli oggetti), `supervision` (tracking + disegno dei box), `scikit-learn` (per separare le squadre dai colori). Richiede ~1 minuto.

In [ ]:
!pip -q install ultralytics supervision scikit-learn
print('\n✅ Installazione completata')

## 2) Carica la tua clip
Esegui la cella, clicca su **Scegli file** e seleziona un video (`.mp4`).

In [ ]:
from google.colab import files
import os

print('Seleziona la clip da caricare...')
uploaded = files.upload()
SOURCE_VIDEO = list(uploaded.keys())[0]
print(f'\n✅ Caricato: {SOURCE_VIDEO}')
print(f'   Dimensione: {os.path.getsize(SOURCE_VIDEO)/1e6:.1f} MB')

## 3) Carica il modello di riconoscimento
Usiamo **YOLOv8**, un modello già addestrato che riconosce persone e palloni. È il punto di partenza: funziona subito, senza chiavi né configurazioni.

> Più avanti lo sostituiremo con un modello **specifico per il calcio** (che distingue giocatore / portiere / arbitro / palla in modo molto più preciso). Per ora questo basta per vedere il sistema in azione.

In [ ]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Calcolo su: {device.upper()}  (se dice CPU, attiva la GPU dal menu Runtime!)')

# yolov8x = versione grande/precisa. Su GPU va bene.
model = YOLO('yolov8x.pt')

# Nel modello COCO: classe 0 = persona, classe 32 = pallone sportivo
PERSON_ID = 0
BALL_ID = 32
print('✅ Modello pronto')

## 4) Funzione per assegnare le squadre dal colore della maglia
Ritagliamo la parte alta del corpo di ogni giocatore (la maglia) e usiamo il colore medio. Con un algoritmo (**KMeans**) raggruppiamo i giocatori in **2 squadre** in base al colore. Semplice ma efficace per iniziare.

In [ ]:
import numpy as np
import cv2
from sklearn.cluster import KMeans

def colore_maglia(frame, box):
    """Estrae il colore medio della maglia da un box [x1,y1,x2,y2]."""
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    # Prendiamo la fascia tra il 15% e il 45% dell'altezza = il busto/maglia
    ty1, ty2 = y1 + int(0.15*h), y1 + int(0.45*h)
    crop = frame[max(0,ty1):max(0,ty2), max(0,x1):max(0,x2)]
    if crop.size == 0:
        return np.array([0, 0, 0])
    return crop.reshape(-1, 3).mean(axis=0)

class AssegnatoreSquadre:
    """Impara i 2 colori squadra dai primi frame, poi classifica ogni giocatore."""
    def __init__(self):
        self.kmeans = None
    def allena(self, colori):
        self.kmeans = KMeans(n_clusters=2, n_init=10, random_state=42).fit(colori)
    def squadra(self, colore):
        return int(self.kmeans.predict([colore])[0]) if self.kmeans else 0

print('✅ Funzioni squadre pronte')

## 5) Elaborazione del video (il cuore del programma)
Per ogni fotogramma: rileviamo persone e palla → seguiamo ogni giocatore con un ID stabile (**ByteTrack**) → assegniamo la squadra dal colore → disegniamo tutto. Il risultato viene salvato in `output_annotato.mp4`.

Una clip di 30s richiede in genere **1–3 minuti** su GPU.

In [ ]:
import supervision as sv

TARGET_VIDEO = 'output_annotato.mp4'
info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
print(f'Video: {info.width}x{info.height}, {info.total_frames} frame, {info.fps} fps')

tracker = sv.ByteTrack(frame_rate=info.fps)

# Colori per disegnare: squadra 0, squadra 1, palla, arbitro/altro
COLORI = sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#FFFFFF'])
box_annot = sv.BoxAnnotator(color=COLORI)
label_annot = sv.LabelAnnotator(color=COLORI, text_scale=0.5, text_thickness=1)
ball_annot = sv.TriangleAnnotator(color=sv.Color.from_hex('#FFD700'), base=20, height=18)

assegnatore = AssegnatoreSquadre()
colori_raccolti = []
FRAME_CALIBRAZIONE = 30  # primi 30 frame per imparare i colori squadra

frame_gen = sv.get_video_frames_generator(SOURCE_VIDEO)

with sv.VideoSink(TARGET_VIDEO, info) as sink:
    for i, frame in enumerate(frame_gen):
        result = model(frame, verbose=False, imgsz=1280)[0]
        det = sv.Detections.from_ultralytics(result)

        # Separiamo palla e persone
        palla = det[det.class_id == BALL_ID]
        persone = det[det.class_id == PERSON_ID]
        persone = tracker.update_with_detections(persone)

        # Fase di calibrazione: raccogliamo i colori maglia
        if i < FRAME_CALIBRAZIONE:
            for box in persone.xyxy:
                colori_raccolti.append(colore_maglia(frame, box))
        elif assegnatore.kmeans is None and len(colori_raccolti) > 4:
            assegnatore.allena(np.array(colori_raccolti))

        # Assegniamo squadra + etichetta ad ogni giocatore
        labels = []
        squadre = []
        for box, tid in zip(persone.xyxy, persone.tracker_id):
            if assegnatore.kmeans is not None:
                sq = assegnatore.squadra(colore_maglia(frame, box))
            else:
                sq = 0
            squadre.append(sq)
            labels.append(f'#{tid} S{sq+1}')
        if len(squadre):
            persone.class_id = np.array(squadre)

        # Disegno
        out = frame.copy()
        out = box_annot.annotate(out, persone)
        out = label_annot.annotate(out, persone, labels=labels)
        if len(palla):
            out = ball_annot.annotate(out, palla)
        sink.write_frame(out)

        if i % 30 == 0:
            print(f'  ...frame {i}/{info.total_frames}')

print(f'\n✅ Fatto! Salvato in: {TARGET_VIDEO}')

## 6) Guarda e scarica il risultato

In [ ]:
from google.colab import files
files.download('output_annotato.mp4')
print('✅ Download avviato. Apri il file sul tuo PC per vedere il risultato.')

---
## ✅ Cosa hai ottenuto e cosa viene dopo

Hai costruito lo **strato 1**: detection + tracking + squadre. È la base di tutto.

**Limiti attuali (normali a questo stadio):**
- Il modello generico a volte confonde palla/persone o perde gli ID quando i giocatori si sovrappongono → si risolve con un **modello specifico per il calcio**.
- I colori squadra possono sbagliare con maglie simili o portieri → si affina dopo.
- Niente distanze in metri ancora → serve la **homography** (Fase 2).

**Prossimo passo che ti propongo:** sostituire YOLOv8 generico con un modello allenato sul calcio (distingue giocatore/portiere/arbitro/palla) e iniziare a salvare le **posizioni su file**, così passiamo dai "box sul video" ai **dati veri** da analizzare. Da lì nasce lo scouting.